In [ ]:
import pandas as pd

In [ ]:
raw_stats = pd.read_csv("../data/raw/team_stats.csv", index_col=0)
raw_sched = pd.read_csv("../data/raw/team_schedules.csv", index_col=0)
print(raw_stats.shape, raw_sched.shape)

In [ ]:
STAT_KEEP = [
    "season", "week", "team", "opponent_team",
    # Passing
    "completions", "attempts", "passing_yards", "passing_tds",
    "passing_interceptions", "sacks_suffered", "sack_yards_lost",
    "passing_air_yards", "passing_yards_after_catch",
    "passing_first_downs", "passing_epa", "passing_cpoe",
    # Rushing
    "carries", "rushing_yards", "rushing_tds", "rushing_first_downs", "rushing_epa",
    # Defense
    "def_tackles_solo", "def_tackles_for_loss", "def_sacks", "def_qb_hits",
    "def_interceptions", "def_interception_yards", "def_pass_defended",
    "def_fumbles_forced",
    # Penalties & timeouts
    "penalties", "penalty_yards", "timeouts",
    # Returns
    "punt_returns", "punt_return_yards", "kickoff_returns", "kickoff_return_yards",
    # Kicking
    "fg_made", "fg_att", "fg_long", "fg_pct",
]

stats = raw_stats[STAT_KEEP].copy()
stats.shape

In [ ]:
SCHED_KEEP = [
    "season", "week", "game_type",
    "away_team", "away_score", "home_team", "home_score",
    "overtime", "away_rest", "home_rest",
    "div_game", "roof", "surface", "temp", "wind",
]

sched = raw_sched[SCHED_KEEP].dropna(subset=["away_score", "home_score"]).copy()
sched.head()

In [ ]:
GAME_COLS = ["season", "week", "overtime", "div_game", "roof", "surface", "temp", "wind"]

home = sched.copy()
home["team"] = home["home_team"]
home["opponent_team"] = home["away_team"]
home["is_home"] = True
home["won"] = home["home_score"] > home["away_score"]
home["rest"] = home["home_rest"]
home["opp_rest"] = home["away_rest"]

away = sched.copy()
away["team"] = away["away_team"]
away["opponent_team"] = away["home_team"]
away["is_home"] = False
away["won"] = away["away_score"] > away["home_score"]
away["rest"] = away["away_rest"]
away["opp_rest"] = away["home_rest"]

TEAM_SCHED_COLS = GAME_COLS + ["game_type", "team", "opponent_team", "is_home", "won", "rest", "opp_rest"]
sched_long = pd.concat([home[TEAM_SCHED_COLS], away[TEAM_SCHED_COLS]])
sched_long.head()

In [ ]:
df = stats.merge(
    sched_long,
    on=["season", "week", "team", "opponent_team"],
    how="inner"
)

df[["team", "opponent_team", "is_home", "won", "rest", "opp_rest"]].head()

In [ ]:
STAT_COLS = [c for c in STAT_KEEP if c not in ("season", "week", "team", "opponent_team")]

opp = df[["season", "week", "team", "opponent_team"] + STAT_COLS].copy()
opp.columns = ["season", "week", "opp_team", "opp_opponent"] + [f"opp_{c}" for c in STAT_COLS]

full = df.merge(
    opp,
    left_on=["season", "week", "team", "opponent_team"],
    right_on=["season", "week", "opp_opponent", "opp_team"],
    how="inner"
).drop(columns=["opp_team", "opp_opponent"])

full.shape

In [ ]:
full = full[full["game_type"] == "REG"].drop(columns="game_type")
full = full.sort_values(["season", "week", "team"]).reset_index(drop=True)
full.head()

In [ ]:
full.to_csv("../data/processed/team_stats_processed.csv", index=False)
print(f"Saved {len(full)} rows, {len(full.columns)} columns")